# Koala-Kangaroo YOLO

## Logging And Setup

We setup our logging using `wandb`
We also initialize our global variables

In [ ]:
import os
os.environ["WANDB_DISABLE_SSL"] = "true"
os.environ["ALBUMENTATIONS_DISABLE"] = "1" # Must be set before YOLO import
from ultralytics import settings
import yaml as pyyaml
settings.update({"wandb": True,
                 "clearml": False,
                 "comet": False})
YOLO_MODEL = "yolo11s.pt"
PROJECT_NAME = "yolo-koala-kangaroo"
EXPERIMENT_NAME = "1_original_20"
EXPERIMENT_CONFIG = f"experiments/{EXPERIMENT_NAME}.yaml"
with open(EXPERIMENT_CONFIG) as f:
    EXPERIMENT_PARAMS = pyyaml.safe_load(f)

TEST_VIDEO_PATH = "test_media/test_video.mp4"
TEST_IMAGE_PATH = ["test_media/test_image.jpg","test_media/image_0010_together.jpg", "test_media/image_0011_together.jpg", "test_media/image_0012_together.jpg"]
DATASET_PATH = EXPERIMENT_PARAMS['data']
BEST_MODEL_PATH = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"
SAVED_MODEL_PATH = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best_int8_openvino_model" #koala-kangaroo/exp1/weights/best_int8_openvino_model/
RESULTS_DIR = f"results/{EXPERIMENT_NAME}/"
FINAL_MODEL_NAME = f"{RESULTS_DIR}/{EXPERIMENT_NAME}_int8_openvino_model.zip" 
os.makedirs(RESULTS_DIR, exist_ok=True)


## Training

We specify the training path for our dataset.

For batch size, given we are training in our laptop and faced GPU out of memory problem, we leave it to Yolo to auto batch using -1 which seemed to solve the memory problem.


In [ ]:
from ultralytics import YOLO
from ultralytics import settings

model = YOLO(YOLO_MODEL)  # Load a pre-trained YOLO model
result = model.train(data=DATASET_PATH,
                     save_period=1, # save every epoch
                     batch=-1, # auto batch size, previously set to 16,64 and caused us to crash for GPU memory reasons
                     device=0, # use GPU 0
                     project=PROJECT_NAME, # set project name  for logging in wandb
                     name=EXPERIMENT_NAME, # set experiment name for logging in wandb
                     cfg=EXPERIMENT_CONFIG, # use yolov11n config
                     plots=True)

## Validation

Here we are looking for mAP50 and mAP50-95 score

In [ ]:
model = YOLO(BEST_MODEL_PATH)
metrics = model.val(data=DATASET_PATH, device="0")

# Retrieve precision and recall
precision = metrics.box.mp  # Mean Precision
recall = metrics.box.mr     # Mean Recall

# Calculate the F1 score
print(metrics)
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
    print(f"F1 Score: {f1_score}")
else:
    print("Precision and recall are zero, cannot calculate F1 score.")
print(f"Precision: {precision}, Recall: {recall}")
print("Validation complete.")

## Export

In [ ]:
model = YOLO(BEST_MODEL_PATH)
exported_path = model.export(format="openvino", int8=True)

In [ ]:
from posixpath import basename
import zipfile
zip = zipfile.ZipFile(FINAL_MODEL_NAME, "w", zipfile.ZIP_DEFLATED)
for file_name in os.listdir(exported_path):
    full_path = os.path.join(exported_path, file_name)
    if os.path.isfile(full_path):
        zip.write(full_path, arcname=basename(file_name))
zip.close()
print(f"Exported INT8 OpenVINO model to {FINAL_MODEL_NAME}")

## Inference

In [ ]:
import ultralytics
from ultralytics import YOLO
from PIL import Image

source = TEST_IMAGE_PATH
model = YOLO(BEST_MODEL_PATH, task='detect')
result = model(source, conf=0.5, iou=0.6)

# Visualize the results
for i, r in enumerate(result):
    print(r)
    # Plot results image
    im_bgr = r.plot()  # BGR-order numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image

    # Show results to screen (in supported environments)
    r.show()

    # Save results to disk
    r.save(filename=f"{RESULTS_DIR}/results-{EXPERIMENT_NAME}-{i}.jpg")

## Video

In [ ]:
from ultralytics import YOLO
import cv2
# Load the YOLO model
model = YOLO(SAVED_MODEL_PATH, task="detect")
# model = YOLO("exp4.2_int8_openvino_model", task="detect")
#exp4.2_int8_openvino_model
# Open the video file
video_path = TEST_VIDEO_PATH
cap = cv2.VideoCapture(video_path)

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO inference on the frame on GPU Device 0
        results = model(frame, conf=0.6, device="cpu")

        # Visualize the results on the frame
        annotated_frame = results[0].plot()

        # Display the annotated frame
        cv2.imshow("YOLO Inference", annotated_frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

## Write Video

In [ ]:
from ultralytics import YOLO
import cv2
# from tqdm import tqdm
from tqdm.auto import tqdm

def write_video(video_in_filepath, video_out_filepath, model):
    # Open the video file

    video_reader = cv2.VideoCapture(video_in_filepath)

    nb_frames = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_h = max(640,int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    frame_w = max(640, int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH)))
    fps = video_reader.get(cv2.CAP_PROP_FPS)

    video_writer = cv2.VideoWriter(video_out_filepath,
                            cv2.VideoWriter_fourcc(*'mp4v'),
                            fps,
                            (frame_w, frame_h))

    # Loop through the video frames
    for i in tqdm(range(nb_frames)):
        # Read a frame from the video
        success, frame = video_reader.read()

        if success:
            # Run YOLO inference on the frame on GPU Device 0
            results = model(frame, conf=0.6, device=0)

            # Visualize the results on the frame
            annotated_frame = results[0].plot()

            # Write the annotated frame
            video_writer.write(annotated_frame)

    video_reader.release()
    video_writer.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

In [ ]:
from pathlib import Path
import os

video_in_file = TEST_VIDEO_PATH
basename = Path(video_in_file).stem
video_out_file = os.path.join(RESULTS_DIR,basename + '_detected' + '.mp4')
model = YOLO(SAVED_MODEL_PATH, task="detect")
write_video(video_in_file, video_out_file, model)